# DocuFit AI 보완 상담 Agent

DocuFit은 기업 담당자가 기업대출 서류를 은행에 접수하기 전에 누락·불일치·만료 여부를 점검하는 B2B 서비스입니다. 이 노트북은 DocuFit 전체 기능 중 **OCR 이후의 서류 검증과 보완 상담 기능**을 구현한 LangChain 기반 프로토타입입니다.

- **RAG**: 기업대출 서류 제출 규정 검색
- **Tool Calling Agent**: 질문에 따라 서류 검증 또는 규정 검색 도구 선택
- **Memory**: 같은 세션의 후속 질문 문맥 유지
- **Gradio**: 대화형 데모 UI

> 실제 OCR 대신 OCR 추출 결과를 모의 데이터로 사용합니다. Agent는 서류 오류를 임의로 판단하지 않고 검증 Tool과 제출 규정 검색 Tool의 결과를 근거로 답변하며, 대출 승인과 신용평가는 수행하지 않습니다.

## 서비스 목적과 범위

### 에이전트의 목적
이 에이전트는 DocuFit 전체 시스템 중 **OCR 필드 추출 이후의 서류 검증과 보완 상담**을 담당합니다. 기업 담당자가 단순히 `검증 실패`만 확인하는 데서 그치지 않고, **어느 서류의 어떤 값이 문제인지, 어떤 규정에 따라 왜 보완해야 하는지, 무엇을 수정하거나 재발급해야 하는지**를 대화로 안내합니다.

### 주요 사용자
- **기업 담당자**: 현재 신청 건의 오류와 필요한 조치를 질문합니다.
- **DocuFit 시스템**: OCR 검증 결과와 제출 규정을 Agent에 제공합니다.

### 구현 범위와 가정
- 포함: 모의 OCR 결과, 규칙 기반 검증, 제출 규정 RAG, Tool Calling Agent, 연속대화 Memory, Gradio UI
- 가정: 제출 서류는 OCR로 필드가 추출되었고, 은행의 제출 규정도 검색 가능한 텍스트로 등록되어 있음
- 제외: 실제 OCR 연동, 규정 PDF 텍스트화·갱신, 대출 승인, 신용평가, 영속 DB 저장

## DocuFit에서 이번 구현이 담당하는 부분

```text
[DocuFit 전체]
서류 업로드 → OCR 추출 → [오류 검증 → 검증 결과·보완 상담] → 보완·재제출 → 은행 검토
                         └────── 이번 구현 범위 ──────┘

[이번 프로토타입]
모의 OCR 결과 → 검증 Tool ───────────────┐
                                         ├→ Agent 답변 → Memory 저장 → 후속 질문
사용자 질문 → Agent → 규정 검색 RAG Tool ┘
```

### 에이전트 행동 시나리오

1. 기업 담당자가 `현재 어떤 오류가 있어?`라고 묻습니다.
2. Agent는 `check_documents` Tool을 호출해 모의 OCR 추출값을 기업 기준정보·유효기간 규칙과 비교하고 오류 문서와 원인을 확인합니다.
3. 사용자가 `왜 보완해야 해?`라고 물으면 `search_submission_rules` Tool이 사전에 텍스트로 등록된 은행 규정에서 근거를 검색합니다.
4. Agent는 검증 결과와 규정 검색 결과를 종합해 오류 원인·규정 근거·보완 행동을 기업 담당자에게 안내합니다.
5. 사용자가 `그럼 어떻게 보완해야 해?`처럼 문서명을 생략해도 Memory가 앞선 대화의 `납세증명서`를 기억해 연속해서 답변합니다.

> **한 줄 요약:** OCR 기반 서류 검증 결과를 바탕으로, 기업 담당자에게 오류의 원인·규정 근거·보완 방법을 연속 대화로 안내하는 Agent입니다.

## Tool Calling 설계

Agent가 모든 답을 자체 지식으로 생성하지 않도록, 업무 성격이 다른 기능을 두 개의 Tool로 분리했습니다.

| Tool | 사용하는 경우 | 처리 방식 | 반환 내용 |
|---|---|---|---|
| `check_documents` | 현재 신청 건의 오류·보완 대상을 묻는 경우 | 모의 OCR 결과를 Python 규칙으로 비교 | 오류 코드, 대상 문서, RULE ID, 원인, 조치 |
| `search_submission_rules` | 제출 기준·유효기간·오류 근거를 묻는 경우 | 질문을 임베딩해 Vector Store에서 관련 규정 검색 | RULE ID, 규정 제목, 관련 내용 |

### 질문에 따른 동적 선택

```text
‘현재 어떤 오류가 있어?’           → check_documents
‘납세증명서 유효기간 기준은?’       → search_submission_rules
‘왜 보완해야 하고 어떻게 고쳐?’     → 두 Tool을 함께 사용
‘그 서류가 뭐였지?’                → Tool 없이 이전 대화 Memory 활용
```

값 비교와 날짜 판정은 검증 Tool이 담당하고, 관련 규정 검색은 RAG Tool이 담당합니다. Agent는 질문에 맞는 Tool을 선택하고 두 결과를 기업 담당자가 이해할 수 있는 답변으로 정리합니다.

## 0. 가상환경 생성 및 Jupyter 커널 선택

노트북을 처음 실행할 때는 VS Code에서 이 파일이 있는 폴더를 열고, **터미널**에 다음 명령을 순서대로 입력합니다.

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip ipykernel
python -m ipykernel install --user --name docufit --display-name "Python (DocuFit)"
```

### VS Code에서 커널 선택

1. 이 노트북을 열고 오른쪽 위의 **커널 선택**를 누릅니다.
2. **Jupyter 커널** → **Python (DocuFit)**을 선택합니다.
3. 목록에 없다면 **Python 환경**에서 `.venv/bin/python`을 선택합니다.
4. 아래 확인 코드를 실행했을 때 경로가 `.venv/bin/python`으로 끝나면 정상입니다.

```python
import sys
print(sys.executable)
```

### API 키 설정

노트북과 같은 폴더의 `.env`에 다음과 같이 입력합니다. `.env`는 API 키가 들어 있으므로 제출하거나 Git에 커밋하지 않습니다.

```env
OPENAI_API_KEY=발급받은_API_키
LLM_MODEL=gpt-4o-mini
```

## 1. 패키지 설치 및 환경 설정
위에서 **Python (DocuFit)** 커널을 선택한 뒤 첫 번째 코드 셀을 실행해 필요한 패키지를 설치합니다. 별도 `requirements.txt`는 사용하지 않으며, API 키는 같은 폴더의 `.env`에서 불러옵니다. 설치가 끝난 후 커널 재시작 안내가 나오면 한 번 재시작합니다.

In [1]:
# 최초 실행 시 이 셀을 한 번 실행하세요. 현재 노트북 커널에 설치됩니다.
%pip install -q "langchain>=1.3,<2" "langchain-openai>=1.4,<2" "langchain-text-splitters>=1,<2" "langgraph>=1,<2" "gradio>=5,<7" "pydantic>=2,<3" "python-dotenv>=1,<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from datetime import date, datetime
from typing import Optional
from uuid import uuid4

import gradio as gr
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(dotenv_path=".env")  # 현재 노트북과 같은 폴더의 .env 로드

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY 환경변수를 설정해 주세요.")

MODEL_NAME = os.getenv("LLM_MODEL", "gpt-4o-mini")
SUBMISSION_DATE = date(2026, 9, 15)  # 재현 가능한 데모를 위한 기준일
print(f"모델: {MODEL_NAME} / 제출 기준일: {SUBMISSION_DATE}")

모델: gpt-4o-mini / 제출 기준일: 2026-09-15


## 2. 모의 OCR 결과와 구조화 스키마
실제 DocuFit에서는 OCR API의 응답이 이 스키마로 들어옵니다. 이번 노트북에서는 동일한 형태의 모의 결과를 사용합니다.

In [3]:
class DocumentRecord(BaseModel):
    document_type: str = Field(description="문서 유형 코드")
    document_name: str = Field(description="사용자에게 표시할 문서명")
    company_name: str = Field(description="OCR로 추출한 기업명")
    business_no: str = Field(description="OCR로 추출한 사업자등록번호")
    issue_date: Optional[str] = Field(default=None, description="발급일 YYYY-MM-DD")
    valid_until: Optional[str] = Field(default=None, description="유효기간 YYYY-MM-DD")


class OCRPackage(BaseModel):
    application_id: str
    expected_company_name: str
    expected_business_no: str
    documents: list[DocumentRecord]


mock_ocr_result = {
    "application_id": "APP-2026-0915-001",
    "expected_company_name": "두루미테크 주식회사",
    "expected_business_no": "123-45-67890",
    "documents": [
        {
            "document_type": "BUSINESS_REGISTRATION",
            "document_name": "사업자등록증",
            "company_name": "두루미테크 주식회사",
            "business_no": "123-45-67890",
            "issue_date": "2026-07-01",
            "valid_until": None,
        },
        {
            "document_type": "TAX_CERTIFICATE",
            "document_name": "납세증명서",
            "company_name": "두루미테크 주식회사",
            "business_no": "123-45-67891",
            "issue_date": "2026-07-31",
            "valid_until": "2026-08-31",
        },
    ],
}

ocr_package = OCRPackage.model_validate(mock_ocr_result)
print(ocr_package.model_dump_json(indent=2))

{
  "application_id": "APP-2026-0915-001",
  "expected_company_name": "두루미테크 주식회사",
  "expected_business_no": "123-45-67890",
  "documents": [
    {
      "document_type": "BUSINESS_REGISTRATION",
      "document_name": "사업자등록증",
      "company_name": "두루미테크 주식회사",
      "business_no": "123-45-67890",
      "issue_date": "2026-07-01",
      "valid_until": null
    },
    {
      "document_type": "TAX_CERTIFICATE",
      "document_name": "납세증명서",
      "company_name": "두루미테크 주식회사",
      "business_no": "123-45-67891",
      "issue_date": "2026-07-31",
      "valid_until": "2026-08-31"
    }
  ]
}


## 3. 서류 검증 Tool
값 비교와 날짜 계산은 LLM이 아닌 Python 규칙으로 처리합니다. 현재 데모에는 **필수서류 누락**, **사업자등록번호 불일치**, **유효기간 만료** 검사를 포함합니다.


In [4]:
REQUIRED_DOCUMENTS = {
    "BUSINESS_REGISTRATION": "사업자등록증",
    "TAX_CERTIFICATE": "납세증명서",
    "FINANCIAL_STATEMENT": "재무제표",
}


def _normalize_business_no(value: str) -> str:
    return "".join(character for character in value if character.isdigit())


def validate_package(package: OCRPackage) -> dict:
    issues = []
    uploaded_types = {document.document_type for document in package.documents}

    for document_type, document_name in REQUIRED_DOCUMENTS.items():
        if document_type not in uploaded_types:
            issues.append({
                "code": "MISSING_DOCUMENT",
                "rule_id": "RULE-001",
                "document": document_name,
                "reason": f"필수서류인 {document_name}가 제출되지 않았습니다.",
                "action": f"{document_name}를 준비해 추가 제출하세요.",
            })

    expected_no = _normalize_business_no(package.expected_business_no)
    for document in package.documents:
        if _normalize_business_no(document.business_no) != expected_no:
            issues.append({
                "code": "BUSINESS_NO_MISMATCH",
                "rule_id": "RULE-002",
                "document": document.document_name,
                "reason": f"기준 사업자등록번호와 {document.document_name}의 번호가 다릅니다.",
                "action": "올바른 기업의 서류인지 확인한 뒤 다시 제출하세요.",
            })

        if document.valid_until:
            valid_until = datetime.strptime(document.valid_until, "%Y-%m-%d").date()
            if valid_until < SUBMISSION_DATE:
                issues.append({
                    "code": "EXPIRED_DOCUMENT",
                    "rule_id": "RULE-003",
                    "document": document.document_name,
                    "reason": f"유효기간이 {document.valid_until}에 만료되었습니다.",
                    "action": f"{document.document_name}를 최신본으로 다시 발급하세요.",
                })

    return {
        "application_id": package.application_id,
        "status": "보완 필요" if issues else "제출 가능",
        "submission_date": SUBMISSION_DATE.isoformat(),
        "issues": issues,
    }


@tool
def check_documents() -> str:
    """현재 대출 신청의 OCR 서류 정보를 검사해 누락, 번호 불일치, 만료 오류와 조치 방법을 반환한다. 현재 서류 상태나 보완 대상을 묻는 질문에 사용한다."""
    return json.dumps(validate_package(ocr_package), ensure_ascii=False, indent=2)


print(check_documents.invoke({}))

{
  "application_id": "APP-2026-0915-001",
  "status": "보완 필요",
  "submission_date": "2026-09-15",
  "issues": [
    {
      "code": "MISSING_DOCUMENT",
      "rule_id": "RULE-001",
      "document": "재무제표",
      "reason": "필수서류인 재무제표가 제출되지 않았습니다.",
      "action": "재무제표를 준비해 추가 제출하세요."
    },
    {
      "code": "BUSINESS_NO_MISMATCH",
      "rule_id": "RULE-002",
      "document": "납세증명서",
      "reason": "기준 사업자등록번호와 납세증명서의 번호가 다릅니다.",
      "action": "올바른 기업의 서류인지 확인한 뒤 다시 제출하세요."
    },
    {
      "code": "EXPIRED_DOCUMENT",
      "rule_id": "RULE-003",
      "document": "납세증명서",
      "reason": "유효기간이 2026-08-31에 만료되었습니다.",
      "action": "납세증명서를 최신본으로 다시 발급하세요."
    }
  ]
}


## 4. 제출 규정 RAG 구성
규정마다 `rule_id`를 부여해 최종 답변에서 근거를 확인할 수 있도록 합니다.

In [5]:
rule_documents = [
    Document(
        page_content="기업대출 접수 시 사업자등록증, 납세증명서, 최근 재무제표를 필수로 제출해야 한다. 누락된 서류가 있으면 접수 전에 보완해야 한다.",
        metadata={"rule_id": "RULE-001", "title": "필수 제출서류"},
    ),
    Document(
        page_content="모든 제출서류의 사업자등록번호는 대출 신청 기업의 기준 사업자등록번호와 일치해야 한다.",
        metadata={"rule_id": "RULE-002", "title": "사업자등록번호 일치"},
    ),
    Document(
        page_content="납세증명서는 제출일 현재 유효해야 한다. 유효기간이 지난 경우 최신 납세증명서를 다시 발급해 제출해야 한다.",
        metadata={"rule_id": "RULE-003", "title": "납세증명서 유효기간"},
    ),
    Document(
        page_content="서류의 기업명 또는 사업자등록번호가 다른 경우 잘못된 기업의 서류인지 확인하고, 기준 정보와 일치하는 문서로 교체해야 한다.",
        metadata={"rule_id": "RULE-004", "title": "기업정보 불일치 보완"},
    ),
    Document(
        page_content="DocuFit의 검증 결과는 서류 접수 전 점검을 위한 보조 정보다. 대출 승인 여부와 기업의 신용평가를 결정하지 않는다.",
        metadata={"rule_id": "RULE-005", "title": "서비스 범위"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=20)
chunks = splitter.split_documents(rule_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


def format_retrieved_documents(documents: list[Document]) -> str:
    return "\n".join(
        f"[{document.metadata['rule_id']}] {document.metadata['title']}: {document.page_content}"
        for document in documents
    )


rag_chain = retriever | RunnableLambda(format_retrieved_documents)


@tool
def search_submission_rules(query: str) -> str:
    """기업대출 서류의 필수 제출 항목, 유효기간, 정보 일치, 보완 방법과 관련된 규정을 검색한다. 일반 기준이나 오류의 근거를 묻는 질문에 사용한다."""
    return rag_chain.invoke(query)


print(search_submission_rules.invoke({"query": "납세증명서가 만료되면 어떻게 해야 하나요?"}))

[RULE-003] 납세증명서 유효기간: 납세증명서는 제출일 현재 유효해야 한다. 유효기간이 지난 경우 최신 납세증명서를 다시 발급해 제출해야 한다.
[RULE-001] 필수 제출서류: 기업대출 접수 시 사업자등록증, 납세증명서, 최근 재무제표를 필수로 제출해야 한다. 누락된 서류가 있으면 접수 전에 보완해야 한다.
[RULE-004] 기업정보 불일치 보완: 서류의 기업명 또는 사업자등록번호가 다른 경우 잘못된 기업의 서류인지 확인하고, 기준 정보와 일치하는 문서로 교체해야 한다.


## 5. Tool Calling Agent와 세션 Memory
Agent는 질문을 해석해 필요한 Tool을 직접 선택합니다. `thread_id`가 같은 호출은 이전 대화를 기억합니다.

In [6]:
model = init_chat_model(
    MODEL_NAME,
    model_provider="openai",
    temperature=0,
)
checkpointer = InMemorySaver()

SYSTEM_PROMPT = """
당신은 기업대출 서류를 은행에 제출하기 전 오류와 보완 방법을 안내하는 DocuFit 상담 Agent다.

- 현재 신청 건의 실제 오류나 보완 대상을 묻는 경우 check_documents를 사용한다.
- 제출 기준, 유효기간 또는 오류의 제도적 근거를 묻는 경우 search_submission_rules를 사용한다.
- '왜 보완해야 하는지와 어떻게 고치는지'를 함께 묻는 경우 두 도구를 모두 사용한다.
- 도구 결과에 없는 오류나 규정을 만들지 않는다.
- 한 번의 사용자 요청에서 같은 도구는 최대 한 번만 호출한다. 여러 규정이 필요하면 하나의 검색어로 묶어서 검색한다.
- '그 서류', '그 오류'처럼 앞선 대상을 가리키는 후속 질문에는 해당 대상만 답하고 다른 오류를 다시 나열하지 않는다.
- 규정을 근거로 답할 때는 RULE ID를 함께 표시한다. 각 오류에는 check_documents가 반환한 rule_id를 그대로 사용하고 다른 ID로 바꾸지 않는다.
- 오류 위치, 원인, 사용자가 취할 행동을 한국어로 간결하게 설명한다.
- 대출 승인 여부나 신용평가 결과는 판단하지 않는다.
"""

agent = create_agent(
    model=model,
    tools=[check_documents, search_submission_rules],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
    middleware=[
        ToolCallLimitMiddleware(tool_name="check_documents", run_limit=1),
        ToolCallLimitMiddleware(tool_name="search_submission_rules", run_limit=1),
    ],
)


def run_agent(question: str, session_id: str = "docufit-demo") -> tuple[str, list[str]]:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": session_id}},
    )

    messages = result["messages"]
    last_user_index = max(
        index for index, message in enumerate(messages)
        if isinstance(message, HumanMessage)
    )
    used_tools = []
    for message in messages[last_user_index + 1:]:
        for tool_call in getattr(message, "tool_calls", []) or []:
            used_tools.append(tool_call["name"])
    used_tools = list(dict.fromkeys(used_tools))  # 같은 Tool의 중복 요청은 한 번만 표시

    return messages[-1].content, used_tools


answer, used_tools = run_agent("현재 서류에 어떤 오류가 있어?", session_id="tool-demo-1")
print("사용 도구:", used_tools)
print(answer)

사용 도구: ['check_documents']
현재 서류에는 다음과 같은 오류가 있습니다:

1. **재무제표 (RULE-001)**
   - **위치**: 제출 서류
   - **원인**: 필수서류인 재무제표가 제출되지 않았습니다.
   - **조치**: 재무제표를 준비해 추가 제출하세요.

2. **납세증명서 (RULE-002)**
   - **위치**: 제출 서류
   - **원인**: 기준 사업자등록번호와 납세증명서의 번호가 다릅니다.
   - **조치**: 올바른 기업의 서류인지 확인한 뒤 다시 제출하세요.

3. **납세증명서 (RULE-003)**
   - **위치**: 제출 서류
   - **원인**: 유효기간이 2026-08-31에 만료되었습니다.
   - **조치**: 납세증명서를 최신본으로 다시 발급하세요.


## 6. 동적 Tool 선택 확인
세 질문을 서로 다른 세션에서 실행해, 질문에 따라 Tool 호출이 달라지는지 확인합니다.

In [7]:
test_questions = [
    "현재 제출한 서류에 어떤 오류가 있어?",
    "납세증명서의 유효기간 기준이 무엇인지 알려줘.",
    "왜 보완이 필요하고, 각각 어떻게 고쳐야 해?",
]

for index, question in enumerate(test_questions, start=1):
    answer, used_tools = run_agent(question, session_id=f"dynamic-test-{index}")
    print(f"\n[질문 {index}] {question}")
    print("[사용 도구]", " -> ".join(used_tools) if used_tools else "도구 미사용")
    print("[답변]", answer)


[질문 1] 현재 제출한 서류에 어떤 오류가 있어?
[사용 도구] check_documents
[답변] 현재 제출한 서류에는 다음과 같은 오류가 있습니다:

1. **재무제표 (RULE-001)**: 필수서류인 재무제표가 제출되지 않았습니다.  
   - **조치**: 재무제표를 준비해 추가 제출하세요.

2. **납세증명서 (RULE-002)**: 기준 사업자등록번호와 납세증명서의 번호가 다릅니다.  
   - **조치**: 올바른 기업의 서류인지 확인한 뒤 다시 제출하세요.

3. **납세증명서 (RULE-003)**: 유효기간이 2026-08-31에 만료되었습니다.  
   - **조치**: 납세증명서를 최신본으로 다시 발급하세요.



[질문 2] 납세증명서의 유효기간 기준이 무엇인지 알려줘.
[사용 도구] search_submission_rules
[답변] 납세증명서의 유효기간 기준은 제출일 현재 유효해야 한다는 것입니다. 만약 유효기간이 지난 경우, 최신 납세증명서를 다시 발급해 제출해야 합니다. (RULE ID: RULE-003)



[질문 3] 왜 보완이 필요하고, 각각 어떻게 고쳐야 해?
[사용 도구] check_documents -> search_submission_rules
[답변] 보완이 필요한 이유와 각 오류에 대한 수정 방법은 다음과 같습니다:

1. **재무제표 (RULE-001)**
   - **이유**: 필수서류인 재무제표가 제출되지 않았습니다.
   - **수정 방법**: 재무제표를 준비해 추가 제출하세요.

2. **납세증명서 (번호 불일치) (RULE-002)**
   - **이유**: 기준 사업자등록번호와 납세증명서의 번호가 다릅니다.
   - **수정 방법**: 올바른 기업의 서류인지 확인한 뒤 다시 제출하세요.

3. **납세증명서 (만료) (RULE-003)**
   - **이유**: 유효기간이 2026-08-31에 만료되었습니다.
   - **수정 방법**: 납세증명서를 최신본으로 다시 발급하세요.


## 7. Memory 확인
두 번째 질문에는 문서명이 없지만 동일한 `thread_id`를 사용하므로 앞선 대화의 대상을 이어서 답할 수 있습니다.

In [8]:
memory_session = "memory-demo"

first_answer, first_tools = run_agent(
    "현재 만료된 서류가 무엇인지 확인해줘.",
    session_id=memory_session,
)
second_answer, second_tools = run_agent(
    "그럼 그 서류는 어떻게 보완해야 해?",
    session_id=memory_session,
)

print("[첫 번째 답변]", first_answer)
print("[첫 번째 사용 도구]", first_tools)
print("\n[후속 답변]", second_answer)
print("[후속 질문 사용 도구]", second_tools)

[첫 번째 답변] 현재 만료된 서류는 **납세증명서**입니다. 

- **위치**: 납세증명서
- **원인**: 유효기간이 2026-08-31에 만료되었습니다.
- **조치 방법**: 납세증명서를 최신본으로 다시 발급하세요. 

이와 관련된 규정은 RULE ID: RULE-003입니다.
[첫 번째 사용 도구] ['check_documents']

[후속 답변] 납세증명서를 보완하는 방법은 다음과 같습니다:

- **위치**: 납세증명서
- **원인**: 유효기간이 지난 경우
- **조치 방법**: 최신 납세증명서를 다시 발급해 제출해야 합니다. 

이와 관련된 규정은 RULE ID: RULE-003입니다.
[후속 질문 사용 도구] ['search_submission_rules']


## 8. 전체 기능 시연 시나리오

아래 질문을 **같은 Gradio 화면에서 순서대로** 입력하면 검증, RAG, 동적 Tool 선택, Memory와 서비스 범위 제한을 모두 확인할 수 있습니다.

| 순서 | 시연 질문 | 확인할 기능 | 예상 Tool |
|---|---|---|---|
| 1 | `현재 제출한 서류에 어떤 오류가 있어?` | 규칙 기반 서류 검증 | `check_documents` |
| 2 | `납세증명서의 유효기간 기준만 알려줘.` | RAG 규정 검색 | `search_submission_rules` |
| 3 | `왜 보완이 필요하고 각각 어떻게 고쳐야 해?` | 오류와 근거 종합, 복수 Tool 선택 | 두 Tool 모두 |
| 4 | `그중 다시 발급해야 하는 서류가 뭐였지?` | 앞선 대화를 참조하는 연속대화 | Memory, Tool 미사용 |
| 5 | `이 결과면 대출 승인이 가능한가?` | 서비스 범위 제한 | Tool 미사용, 승인 판단 거절 |

화면의 **이번 요청에 사용한 Tool** 영역에서 질문에 따라 실제 호출 도구가 달라지는지 확인합니다.

## 9. Gradio UI
처음 실행한 사용자를 위한 시연 가이드와 클릭 가능한 추천 질문을 제공하고, 답변과 함께 실제 사용 Tool을 표시합니다.

In [9]:
INITIAL_SUGGESTIONS = [
    "현재 제출한 서류에 어떤 오류가 있어?",
    "납세증명서의 유효기간 기준만 알려줘.",
    "왜 보완이 필요하고 각각 어떻게 고쳐야 해?",
]


def recommend_followups(question, used_tools):
    """직전 질문과 Tool 실행 결과에 맞는 다음 질문을 추천합니다."""
    tool_set = set(used_tools)
    if tool_set == {"check_documents"}:
        return [
            "왜 이런 오류가 발생했는지 규정과 함께 설명해줘.",
            "그중 다시 발급해야 하는 서류가 뭐야?",
            "누락된 서류부터 보완 순서대로 알려줘.",
        ]
    if tool_set == {"search_submission_rules"}:
        return [
            "그 규정이 현재 내 서류에도 해당하는지 확인해줘.",
            "현재 오류와 함께 보완 방법을 정리해줘.",
            "다시 발급해야 하는 서류가 있는지 알려줘.",
        ]
    if tool_set == {"check_documents", "search_submission_rules"}:
        return [
            "그중 다시 발급해야 하는 서류가 뭐였지?",
            "보완할 작업을 순서대로 정리해줘.",
            "이 결과면 대출 승인이 가능한가?",
        ]
    if "승인" in question:
        return INITIAL_SUGGESTIONS
    return [
        "관련 제출 규정도 알려줘.",
        "현재 남은 보완 사항을 다시 정리해줘.",
        "이전 답변을 보완 순서대로 요약해줘.",
    ]


def submit_question(message, history, session_id):
    """질문을 전송하고, 답변 직후 표시할 후속 질문 버튼을 갱신합니다."""
    if not message or not message.strip():
        return history, "", "### 이번 요청에 사용한 Tool\n질문을 입력해 주세요.", *[gr.update() for _ in range(3)]

    # 같은 브라우저 세션은 동일한 thread_id를 사용하므로 앞선 대화를 기억합니다.
    answer, used_tools = run_agent(message.strip(), session_id=session_id)
    updated_history = list(history or [])
    updated_history.extend([
        {"role": "user", "content": message.strip()},
        {"role": "assistant", "content": answer},
    ])
    tool_label = (
        " → ".join(f"`{name}`" for name in used_tools)
        if used_tools
        else "도구 미사용"
    )
    previous_turns = len(history or []) // 2
    memory_label = (
        f"✅ **Memory 작동** — 이전 대화 {previous_turns}턴을 같은 세션으로 전달했습니다."
        if previous_turns > 0
        else "📝 **Memory 저장 시작** — 이 질문부터 같은 세션에 대화를 저장합니다."
    )
    tool_text = f"### 이번 요청에 사용한 Tool\n{tool_label}\n\n### 대화 Memory\n{memory_label}"
    suggestions = recommend_followups(message, used_tools)
    button_updates = [gr.update(value=suggestion, visible=True) for suggestion in suggestions]
    return updated_history, "", tool_text, *button_updates


def submit_suggestion(selected_question, history, session_id):
    """추천 질문 버튼을 누르면 별도 전송 없이 즉시 질문합니다."""
    return submit_question(selected_question, history, session_id)


def reset_conversation():
    """화면과 Agent Memory를 함께 새 세션으로 초기화합니다."""
    button_updates = [gr.update(value=suggestion, visible=True) for suggestion in INITIAL_SUGGESTIONS]
    return [], "", str(uuid4()), "### 이번 요청에 사용한 Tool\n아직 호출되지 않았습니다.", *button_updates


# 이 셀을 다시 실행할 때 이전 Gradio 서버를 닫아 오래된 화면이 남지 않게 합니다.
if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass

with gr.Blocks(title="DocuFit AI 보완 상담") as demo:
    session_state = gr.State(lambda: str(uuid4()))
    gr.Markdown("# DocuFit AI 보완 상담 Agent")
    gr.Markdown(
        "OCR 검증이 완료된 기업대출 서류에 대해 오류 원인과 보완 방법을 질문해 보세요. "
        "대출 승인과 신용평가는 제공하지 않습니다."
    )
    with gr.Accordion("처음이라면 여기를 열어 시연 순서를 확인하세요", open=True):
        gr.Markdown(
            """
            **추천 시연 순서**  
            1. 현재 오류 확인 → 검증 Tool  
            2. 유효기간 기준 확인 → RAG Tool  
            3. 오류 원인과 보완 방법 요청 → 두 Tool 모두  
            4. ‘그중 다시 발급할 서류가 뭐였지?’ → 연속대화 Memory  
            5. 대출 승인 가능 여부 질문 → 서비스 범위 제한
            """
        )
    chatbot = gr.Chatbot(label="상담 내용", height=430)
    gr.Markdown("### 추천 후속 질문")
    with gr.Row():
        suggestion_1 = gr.Button(INITIAL_SUGGESTIONS[0], variant="secondary")
        suggestion_2 = gr.Button(INITIAL_SUGGESTIONS[1], variant="secondary")
        suggestion_3 = gr.Button(INITIAL_SUGGESTIONS[2], variant="secondary")
    with gr.Row():
        question_box = gr.Textbox(
            placeholder="직접 질문하거나 위 추천 질문을 선택하세요.",
            label="질문",
            scale=8,
        )
        send_button = gr.Button("전송", variant="primary", scale=1)
        reset_button = gr.Button("새 대화", scale=1)
    tool_output = gr.Markdown("### 이번 요청에 사용한 Tool\n아직 호출되지 않았습니다.")

    response_outputs = [chatbot, question_box, tool_output, suggestion_1, suggestion_2, suggestion_3]
    send_button.click(
        fn=submit_question,
        inputs=[question_box, chatbot, session_state],
        outputs=response_outputs,
    )
    question_box.submit(
        fn=submit_question,
        inputs=[question_box, chatbot, session_state],
        outputs=response_outputs,
    )
    for suggestion_button in [suggestion_1, suggestion_2, suggestion_3]:
        suggestion_button.click(
            fn=submit_suggestion,
            inputs=[suggestion_button, chatbot, session_state],
            outputs=response_outputs,
        )
    reset_button.click(
        fn=reset_conversation,
        inputs=None,
        outputs=[chatbot, question_box, session_state, tool_output, suggestion_1, suggestion_2, suggestion_3],
    )

demo.launch(share=False, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## 10. 결과 및 한계

### 구현 결과
- 기업대출 제출 규정 임베딩 및 질문 관련 근거 검색
- 서류 검증·규정 검색 Tool 분리 및 질문별 동적 선택
- 세션 Memory 기반 후속 질문 문맥 유지
- Python 규칙 검증과 LLM 설명 생성 분리를 통한 판정 일관성 확보

### 한계와 향후 개선
- 모의 OCR 결과 사용 → 향후 동일 스키마의 실제 OCR API 응답 연결 필요
- 데모용 규정 데이터 사용 → 운영 시 은행 최신 공식 규정으로 교체 필요
- 인메모리 Memory 사용 → 서버 재시작 시 초기화, 운영 시 영속 Checkpointer 필요
- 서류 검수 보조에 한정 → 대출 승인·신용평가 자동화 범위 제외

### 수업 내용 적용 현황

| 적용 여부 | 수업 내용 | 프로젝트 적용 방식 |
|---|---|---|
| 사용 | Chat Model·Messages | `init_chat_model()`과 메시지 기록을 활용한 상담 답변 생성 |
| 사용 | Agent·Tools·Tool Calling | 질문에 따라 서류 검증·규정 검색 Tool 동적 선택 |
| 사용 | Short-term Memory·Checkpointer | `thread_id`와 `InMemorySaver`를 활용한 세션별 후속 질문 유지 |
| 사용 | Pydantic·Schema | OCR 결과를 `DocumentRecord`, `OCRPackage`로 구조화·검증 |
| 사용 | LCEL·Runnable | `retriever | RunnableLambda` 형태의 RAG 체인 구성 |
| 사용 | RAG | Document 분할, Embedding, Vector Store, Retriever로 제출 규정 검색 |
| 사용 | Middleware | 동일 Tool의 중복 호출 제한 |
| 사용 | Gradio Blocks | 채팅, 추천 질문, Tool·Memory 상태를 표시하는 시연 UI 구성 |
| 일부 사용 | Prompt | System Prompt는 적용했지만 `ChatPromptTemplate`은 미사용 |
| 일부 사용 | Structured Output | Pydantic로 입력 데이터를 구조화했지만 LLM Structured Output은 미사용 |
| 일부 사용 | Document Loader | 규정을 `Document`로 직접 작성, PDF·웹 Loader는 미사용 |
| 미사용 | Streaming·Output Parser·RunnableParallel | 현재 동기식 `invoke()` 기반으로 구성 |
| 미사용 | SQL Toolkit·Sandbox·LangSmith | 현재 프로토타입 범위에서 제외 |